# Import lib & Data path

In [113]:
# !pip3 install transformers torch pandas tqdm sacremoses

In [114]:
from __future__ import annotations

import json
import re
import time
import shutil
import os
import ast
from pathlib import Path

# thư viện toán và ma trận
import numpy as np
import pandas as pd

# Thư viện học máy học sâu
from sklearn.metrics.pairwise import cosine_similarity
import torch
from sentence_transformers import SentenceTransformer
from transformers import MarianMTModel, MarianTokenizer

# Hiện tiến trình
from tqdm import tqdm



from huggingface_hub import login

from dotenv import load_dotenv, find_dotenv# Tự động tìm file .env ở các thư mục cha
load_dotenv(find_dotenv())
login(token=os.getenv("HF_TOKEN"))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Drive connect

In [115]:
# 1. Lấy quyền truy cập Google Drive
# from google.colab import drive
# drive.mount('/content/drive')



def send_file_to_drive(file_path, destination_folder = '/content/drive/MyDrive/movie-recommendation/data/'):
  """
  Gửi file với path sang đích để tránh mất file trước khi chạy xong
  """
  source_file = file_path

  # Tạo thư mục đích nếu nó chưa tồn tại
  os.makedirs(destination_folder, exist_ok=True)

  # Sao chép file
  shutil.copy(source_file, destination_folder)

  print(f"[INFO] Đã sao chép '{source_file}' tới '{destination_folder}' thành công!")


## Config parameters models

In [ ]:
config_dict = {
    'DATA_DIR': Path("../data"),
    'TMDB_DIR': Path("../data/tmdb"),
    'CATALOG_PATH': Path("../data/tmdb/all_movies_fully_translated.csv"),
    'MOVIELENS_DIR': Path("../data/movielens"),
    'RATINGS_PATH': Path("../data/movielens/ratings_filtered_by_movie.csv"),
    'TRAIN_DIR': Path("../data/train"),
    'ARTIFACT_DIR': Path("../data/artifacts/embeddings"),
    'MODEL_NAMES': [
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "sentence-transformers/distiluse-base-multilingual-cased-v2",
    ],
    'BATCH_SIZE': 64,
    'TOP_K': 10,
    'NORMALIZE_EMBEDDINGS': True,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Hệ thống đang chạy trên nền tảng: {DEVICE.upper()}")


class ConfigModel:
    DATA_DIR = config_dict['DATA_DIR']
    TMDB_DIR = config_dict['TMDB_DIR']
    CATALOG_PATH = config_dict['CATALOG_PATH']
    MOVIELENS_DIR = config_dict['MOVIELENS_DIR']
    RATINGS_PATH = config_dict['RATINGS_PATH']
    TRAIN_DIR = config_dict['TRAIN_DIR']
    ARTIFACT_DIR = config_dict['ARTIFACT_DIR']
    MODEL_NAMES = config_dict['MODEL_NAMES']
    BATCH_SIZE = config_dict['BATCH_SIZE']
    TOP_K = config_dict['TOP_K']
    NORMALIZE_EMBEDDINGS = config_dict['NORMALIZE_EMBEDDINGS']
    EVAL_TOP_K = 10

ConfigModel.ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Catalog: {ConfigModel.CATALOG_PATH.resolve()}")
print(f"Artifact directory: {ConfigModel.ARTIFACT_DIR.resolve()}")


# Ví dụ cấu trúc. Thay bằng nhãn đánh giá thật trước khi tính Recall/MRR/nDCG.
EVAL_CASES: list[dict] = []
EVAL_CASES = [
    {"query": "phim phiêu lưu vũ trụ", "relevant_movie_ids": ["603", "157336"]},
]

SEARCH_ID_COLUMN = "tmdb_id"


[INFO] Hệ thống đang chạy trên nền tảng: CPU
Catalog: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\tmdb\all_movies_fully_translated.csv
Artifact directory: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\artifacts\embeddings


# Utils functions

- Load data CSV với path thành biến DataFrame
- Chuyển các cột overview (overview_vi), title (title_vi), genres thành text
- Chuẩn hóa text cơ bản: loại ký tự đặc biệt, dấu cấu, khoảng trắng thừa, chữ hoa thành chữ thường
để text sạch dùng cho các models với xử lý text khi search, embedding, recommendation, ... 

In [117]:
class TextUtils: 
    """Xử lý text cho các model, search text, hỗ trợ load csv với movie catalog"""
    @staticmethod
    def normalize_text(value: str) -> str:
        """
        Chuẩn hóa text cơ bản: loại ký tự đặc biệt, dấu cấu, khoảng trắng thừa, chữ hoa thành chữ thường
        """
        value = re.sub(r"[^\w\s]", " ", str(value or ""))  # loại ký tự đặc biệt
        value = re.sub(r"\s+", " ", str(value or ""))  # loại khoảng trắng thừa
        value = value.strip().lower()  # loại khoảng trắng đầu cuối và chuyển chữ hoa thành chữ thường
        
        return value
    
    @staticmethod
    def create_search_text(title_en = '', 
                           title_vi = '', 
                           overview_en = '', 
                           overview_vi = '', 
                           genres = list[dict] | list[str] | str | None
    ) -> str:
        """
        Tạo cột search_text từ các cột title, overview, genres
        """
        # # log
        # print(f"Title EN: {title_en}, Title VI: {title_vi}")
        # print(f"Overview EN: {overview_en}, Overview VI: {overview_vi}")
        # build title_text
        title_text = (title_vi + " " + title_en)
        title_text = title_text.strip()
        
        # build overview_text
        overview_text = (str(overview_vi) + " " + str(overview_en))
        overview_text = overview_text.strip()

        # build genres_text
        def normalize_genres(value):
            if isinstance(value, str):
                    value = value.strip()
            
                    try:
                        value = json.loads(value)
                    except json.JSONDecodeError:
                        try:
                            value = ast.literal_eval(value)
                        except (ValueError, SyntaxError):
                            return ""
            # print(f"Genres value: {value}, type: {type(value)}")
                        
            if isinstance(value, list): # ['Action', 'Adventure']
                genres_text = ''
                if all(isinstance(g, str) for g in value):
                    genres_text = ', '.join(value)
                elif all(isinstance(g, dict) for g in value):
                    genres_text = ', '.join([g.get('name', '') for g in value])
                else:
                    return ""
                return genres_text.strip()
            return value.strip()
        
        genres_text = normalize_genres(genres)
        
        
        return (
            "Tiêu đề: " + TextUtils.normalize_text(title_text)
            + " Cốt truyện: " + TextUtils.normalize_text(overview_text)
            + " Thể loại: " + TextUtils.normalize_text(genres_text)
        ).strip()
    
    @staticmethod
    def load_csv(path: Path) -> pd.DataFrame:
        """
        Load CSV file to DataFrame
        """
        catalog = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
        
        # check các cột cần thiết trong movie catalog
        required_columns = ["genres", 'title', 'overview', 'title_vi', 'overview_vi', 'tmdb_id', 'movieId']
        columns_missing = [col for col in required_columns if col not in catalog.columns]
        if columns_missing:
            raise KeyError(f"Catalog thiếu các cột cần thiết: {', '.join(columns_missing)}")
        
        # đổi tên movieId thành movie_id 
        catalog.rename(columns={'movieId': 'movie_id'}, inplace=True)
        
        return catalog
    
    @staticmethod
    def add_search_text_column(df: pd.DataFrame) -> pd.DataFrame:
        """
        Thêm cột search_text vào DataFrame
        """
        df["search_text"] = df.apply(
            lambda row: TextUtils.create_search_text(
                title_en=row.get("title", ""),
                title_vi=row.get("title_vi", ""),
                overview_en=row.get("overview", ""),
                overview_vi=row.get("overview_vi", ""),
                genres=row.get("genres", None),
            ),
            axis=1,
        )
        return df[df["search_text"].str.len() > 0].reset_index(drop=True)
    
    
catalog = TextUtils.load_csv(ConfigModel.CATALOG_PATH)
catalog = TextUtils.add_search_text_column(catalog)

print(f"Catalog rows: {len(catalog):,}")

print("Các cột trong catalog: ", catalog.columns.tolist())

print('Duyệt 5 dòng đầu tiên của catalog:')
display(catalog.head(5))

print('Thông tin về các cột trong catalog:')
display(catalog.info())

Catalog rows: 19,706
Các cột trong catalog:  ['tmdb_id', 'title_vi', 'overview_vi', 'genres', 'release_date', 'poster_path', 'adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'homepage', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'movie_id', 'is_ai_translated', 'overview_vi_word_count', 'search_text']
Duyệt 5 dòng đầu tiên của catalog:


,tmdb_id,title_vi,overview_vi,genres,release_date,poster_path,adult,backdrop_path,belongs_to_collection,budget,...,status,tagline,title,video,vote_average,vote_count,movie_id,is_ai_translated,overview_vi_word_count,search_text
0,131232.0,Due amici,"Bạn bè hai tuổi, Nunzio và Pino, có chung một ...","[{'id': 18, 'name': 'Phim Chính Kịch'}]",2002-03-20,/ja7a0XvTUIQ6w1YdvbtytdC1lTF.jpg,False,/r1boovecPQAZo6vQBLZ4NVeRT2g.jpg,NaN,0,...,Released,NaN,Due amici,False,5.400,7,723.0,True,126,Tiêu đề: due amici due amici Cốt truyện: bạn b...
1,79782.0,Wenecja,Một câu chuyện sắp tới kể về một cậu bé tên là...,"[{'id': 18, 'name': 'Phim Chính Kịch'}, {'id':...",2010-06-11,/jk8PwaZ14Duxy8hmLbYLFcYTUP9.jpg,False,/8vrmhLhPucmEOJr6EDxhxzVDMXi.jpg,NaN,1783810,...,Released,NaN,Wenecja,False,7.139,18,895.0,True,34,Tiêu đề: wenecja wenecja Cốt truyện: một câu c...
2,141210.0,The Sleepover,"Thị trấn Derry có một bí mật, nhưng không ai n...","[{'id': 35, 'name': 'Phim Hài'}, {'id': 27, 'n...",2012-10-12,/tdaZD1Db9CXu96oLdAHElJsBPfb.jpg,False,NaN,NaN,0,...,Released,NaN,The Sleepover,False,6.500,11,1115.0,True,20,Tiêu đề: the sleepover the sleepover Cốt truyệ...
3,143750.0,The Farmer's Wife,Khi môi trường của cô bị xâm chiếm bởi người n...,"[{'id': 18, 'name': 'Phim Chính Kịch'}]",2012-06-20,NaN,False,NaN,NaN,0,...,Released,NaN,The Farmer's Wife,False,10.000,1,2223.0,True,87,Tiêu đề: the farmer s wife the farmer s wife C...
4,84198.0,A Place at the Table,"Sử dụng những câu chuyện cá nhân, những tài li...","[{'id': 99, 'name': 'Phim Tài Liệu'}]",2012-03-22,/ptli8HtDrapZdVdBnYSN8zxJtw0.jpg,False,/hdFefzO4RPq4txKEbFU7EulePsS.jpg,NaN,0,...,Released,NaN,A Place at the Table,False,6.593,27,2679.0,True,65,Tiêu đề: a place at the table a place at the t...


Thông tin về các cột trong catalog:
<class 'pandas.DataFrame'>
RangeIndex: 19706 entries, 0 to 19705
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   tmdb_id                 19706 non-null  float64
 1   title_vi                19706 non-null  str    
 2   overview_vi             19008 non-null  str    
 3   genres                  19706 non-null  str    
 4   release_date            19700 non-null  str    
 5   poster_path             18983 non-null  str    
 6   adult                   19706 non-null  bool   
 7   backdrop_path           17093 non-null  str    
 8   belongs_to_collection   2371 non-null   str    
 9   budget                  19706 non-null  int64  
 10  homepage                10 non-null     str    
 11  imdb_id                 16951 non-null  str    
 12  origin_country          19706 non-null  str    
 13  original_language       19706 non-null  str    
 14  original_titl

None

## Evaluation data and metrics

`EVAL_CASES` cần được điền bằng ground truth thật. `relevant_movie_ids` phải dùng cùng loại ID với `SEARCH_ID_COLUMN`. Nếu chưa có nhãn, notebook vẫn chạy search và đo được latency, coverage, uniqueness; không tự tạo Recall/MRR giả.

In [ ]:


class EvaluationMetrics:
    def precision_at_k(self, retrieved: list, relevant: set, k: int) -> float:
        top_items = retrieved[:k]
        return sum(item in relevant for item in top_items) / max(len(top_items), 1)


    def recall_at_k(self, retrieved: list, relevant: set, k: int) -> float:
        if not relevant:
            return 0.0
        return sum(item in relevant for item in retrieved[:k]) / len(relevant)


    def reciprocal_rank(self, retrieved: list, relevant: set) -> float:
        for rank, item in enumerate(retrieved, start=1):
            if item in relevant:
                return 1.0 / rank
        return 0.0


    def ndcg_at_k(self, retrieved: list, relevant: set, k: int) -> float:
        relevance = [int(item in relevant) for item in retrieved[:k]]
        dcg = sum(value / np.log2(rank + 2) for rank, value in enumerate(relevance))
        ideal = sorted(relevance, reverse=True)
        idcg = sum(value / np.log2(rank + 2) for rank, value in enumerate(ideal))
        return dcg / idcg if idcg else 0.0



    def evaluate_results(self, results: pd.DataFrame, eval_cases: list[dict], k: int) -> dict:
        if not eval_cases:
            return {
                "query_count": 0,
                "precision_at_k": None,
                "recall_at_k": None,
                "mrr": None,
                "ndcg_at_k": None,
            }

        metrics = []
        for case in eval_cases:
            query_results = results[results["query"] == case["query"]]
            retrieved = query_results[SEARCH_ID_COLUMN].tolist()
            relevant = set(case["relevant_movie_ids"])
            metrics.append({
                "precision_at_k": self.precision_at_k(retrieved, relevant, k),
                "recall_at_k": self.recall_at_k(retrieved, relevant, k),
                "mrr": self.reciprocal_rank(retrieved, relevant),
                "ndcg_at_k": self.ndcg_at_k(retrieved, relevant, k),
            })

        return {"query_count": len(metrics), **pd.DataFrame(metrics).mean().to_dict()}

# Sentences embedding model

In [119]:
def safe_model_name(model_name: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", model_name)


def embedding_paths(model_name: str) -> tuple[Path, Path, Path]:
    name = safe_model_name(model_name)
    return (
        ConfigModel.ARTIFACT_DIR / f"{name}_embeddings.npy",
        ConfigModel.ARTIFACT_DIR / f"{name}_metadata.csv",
        ConfigModel.ARTIFACT_DIR / f"{name}_config.json",
    )

class SentenceTransformerModel:
    def __init__(self, model_name: str, device: str = DEVICE, catalog_df: pd.DataFrame = catalog):
        self.catalog = catalog_df
        self.set_model(model_name)
        
    def set_model(self, model_name: str):
        self.model_name = model_name
        # for p in embedding_paths(model_name):
        #     print(f"[INFO] Embedding paths: {p.resolve()}")
        self.embedding_path, self.metadata_path, self.config_path = embedding_paths(model_name)
        self.model: SentenceTransformer = SentenceTransformer(
            model_name, 
            device=DEVICE
        )
        self.embeddings = None
        self.metadata = self.catalog[["tmdb_id", "movie_id", "title"]].copy()
        self.config: dict = None
        
    def add_new_texts(self, new_text: str, movie_id: int, tmdb_id: int) -> None:
        """
        Thêm các văn bản mới vào embeddings hiện có và cập nhật metadata.
        """
        if self.embeddings is None or self.metadata is None:
            raise ValueError("Embeddings hoặc metadata chưa được khởi tạo. Vui lòng gọi encode_catalog trước.")

        # Encode các văn bản mới
        new_embeddings = self.model.encode(
            [new_text],
            batch_size=ConfigModel.BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=ConfigModel.NORMALIZE_EMBEDDINGS,
            convert_to_numpy=True,
        ).astype("float32")

        # Cập nhật embeddings hiện có
        self.embeddings = np.vstack([self.embeddings, new_embeddings])

        # Cập nhật metadata (giả sử bạn có một DataFrame metadata mới cho các văn bản mới)
        new_metadata = pd.DataFrame({
            "tmdb_id": [tmdb_id],
            "movie_id": [movie_id],
            "title": [new_text]
        })
        self.metadata = pd.concat([self.metadata, new_metadata], ignore_index=True)

        # Lưu embeddings và metadata cập nhật
        np.save(self.embedding_path, self.embeddings)
        self.metadata.to_csv(self.metadata_path, index=False, encoding="utf-8-sig")

    def encode_catalog(self, text_list: list[str]) -> tuple[np.ndarray, pd.DataFrame, dict]:
        """"""
        print(f"[INFO] Bắt đầu mã hóa văn bản với mô hình: {self.model_name}")
        started_at = time.perf_counter()
        self.embeddings = self.model.encode(
            text_list,
            batch_size=ConfigModel.BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=ConfigModel.NORMALIZE_EMBEDDINGS,
            convert_to_numpy=True,
        ).astype("float32")
        elapsed_seconds = time.perf_counter() - started_at
        print(f"[INFO] Mã hóa hoàn tất trong {elapsed_seconds:.2f} giây")
        
        print(f"[INFO] Lưu embeddings và metadata vào: {self.embedding_path.resolve()} và {self.metadata_path.resolve()}")
        np.save(self.embedding_path, self.embeddings)
        self.metadata.to_csv(self.metadata_path, index=False, encoding="utf-8-sig")
        self.config = {
            "model_name": self.model_name,
            "embedding_dimension": int(self.embeddings.shape[1]),
            "document_count": int(self.embeddings.shape[0]),
            "normalized": ConfigModel.NORMALIZE_EMBEDDINGS,
            "batch_size": ConfigModel.BATCH_SIZE,
            "encode_seconds": elapsed_seconds,
            "text_columns": ["title", "overview", "genres"],
        }
        self.config_path.write_text(json.dumps(self.config, ensure_ascii=False, indent=2), encoding="utf-8")
        
        print(f"[INFO] Kết thúc quá trình mã hóa và lưu trữ embeddings, metadata, và config.")
        return self.embeddings, self.metadata, self.config


# Memory Vector Store

In [120]:
class VectorDbStorage:
    """
    Lưu trữ embeddings và metadata vào ổ đĩa.
    """
    def __init__(self, model_name: str):
        self.model_name = model_name
        # print(f"[INFO] Embedding paths: {embedding_paths(model_name)}")
        self.embedding_path, self.metadata_path, self.config_path = embedding_paths(model_name)
        self.embeddings = None
        self.metadata = None
        self.config: dict = None

    def load(self) -> None:
        """
        Load embeddings và metadata từ ổ đĩa.
        """
        if not self.embedding_path.exists() or not self.metadata_path.exists() or not self.config_path.exists():
            raise FileNotFoundError("Embeddings hoặc metadata hoặc config không tồn tại. Vui lòng chạy encode_catalog trước.")
        
        self.embeddings = np.load(self.embedding_path)
        self.metadata = pd.read_csv(self.metadata_path, encoding="utf-8-sig")
        with open(self.config_path, "r", encoding="utf-8") as f:
            self.config = json.load(f)
            
    def save(self, embeddings: np.ndarray, metadata: pd.DataFrame, config: dict) -> None:
        """
        Lưu embeddings, metadata và config vào ổ đĩa.
        """
        np.save(self.embedding_path, embeddings)
        metadata.to_csv(self.metadata_path, index=False, encoding="utf-8-sig")
        self.config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

# Search and evaluate embedding models

In [121]:
class SearchEngine:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.vector_db = VectorDbStorage(model_name)
        self.vector_db.load()
        self.model = SentenceTransformerModel(model_name)
        self.model.embeddings = self.vector_db.embeddings
        self.model.metadata = self.vector_db.metadata
        self.model.config = self.vector_db.config

    def search(self, query: str, top_k: int) -> tuple[pd.DataFrame, int]:
        
        started_at = time.perf_counter()
        
        query_embedding = self.model.model.encode(
            query,
            convert_to_numpy=True,
            normalize_embeddings=ConfigModel.NORMALIZE_EMBEDDINGS,
        ).astype("float32")

        similarities = cosine_similarity(query_embedding.reshape(1, -1), self.model.embeddings).flatten()
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        elapsed_seconds = time.perf_counter() - started_at
        print(f"[INFO] Tìm kiếm hoàn tất trong {elapsed_seconds:.2f} giây")
        
        results = self.model.metadata.iloc[top_indices].copy()
        results["score"] = similarities[top_indices]
        results["rank"] = range(1, len(results) + 1)
        
        return results[["rank", "tmdb_id", "movie_id", "title", "score"]], elapsed_seconds




# Train model

In [122]:
# lấy 100 dòng
test_catalog = catalog.head(100)
for model_name in ConfigModel.MODEL_NAMES:
    model = SentenceTransformerModel(model_name, device=DEVICE, catalog_df=test_catalog)
    embeddings, metadata, config = model.encode_catalog(test_catalog["search_text"].tolist())
    

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1208.87it/s]


[INFO] Bắt đầu mã hóa văn bản với mô hình: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches: 100%|██████████| 2/2 [00:04<00:00,  2.04s/it]


[INFO] Mã hóa hoàn tất trong 4.09 giây
[INFO] Lưu embeddings và metadata vào: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\artifacts\embeddings\sentence-transformers_paraphrase-multilingual-MiniLM-L12-v2_embeddings.npy và C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\artifacts\embeddings\sentence-transformers_paraphrase-multilingual-MiniLM-L12-v2_metadata.csv
[INFO] Kết thúc quá trình mã hóa và lưu trữ embeddings, metadata, và config.


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 263.19it/s]


[INFO] Bắt đầu mã hóa văn bản với mô hình: sentence-transformers/distiluse-base-multilingual-cased-v2


Batches: 100%|██████████| 2/2 [00:07<00:00,  3.75s/it]

[INFO] Mã hóa hoàn tất trong 7.54 giây
[INFO] Lưu embeddings và metadata vào: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\artifacts\embeddings\sentence-transformers_distiluse-base-multilingual-cased-v2_embeddings.npy và C:\52300070_TMTHUYEN\LEARN_MORE\Apps\dacntt-movie-recommendation\ai-service\data\artifacts\embeddings\sentence-transformers_distiluse-base-multilingual-cased-v2_metadata.csv
[INFO] Kết thúc quá trình mã hóa và lưu trữ embeddings, metadata, và config.


# Seach and eval

In [123]:

QUERY = "phim phiêu lưu và tình bạn trong không gian"
for model_name in ConfigModel.MODEL_NAMES:
    search_engine = SearchEngine(model_name)
    search_results, latency_ms = search_engine.search(QUERY, top_k=5)
    print(f"{model_name}: {latency_ms:.2f} ms")
    display(search_results[["rank", "tmdb_id", "title", "score"]])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1593.97it/s]


[INFO] Tìm kiếm hoàn tất trong 0.03 giây
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: 0.03 ms


,rank,tmdb_id,title,score
67,1,47724.0,Up at the Villa,0.523440
54,2,19214.0,Passion of Mind,0.501612
74,3,9285.0,Con Đường Tình Dục,0.494911
76,4,955.0,Nhiệm Vụ: Bất Khả Thi 2,0.465702
10,5,10472.0,Down to You,0.458396


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 459.08it/s]


[INFO] Tìm kiếm hoàn tất trong 0.03 giây
sentence-transformers/distiluse-base-multilingual-cased-v2: 0.03 ms


,rank,tmdb_id,title,score
98,1,44490.0,Chuck & Buck,0.291561
21,2,2787.0,Riddick: Hành Tinh Tăm Tối,0.242961
0,3,131232.0,Due amici,0.238567
31,4,10416.0,What Planet Are You From?,0.221032
39,5,109479.0,Buddy Boy,0.209160


In [124]:

def benchmark_model(model_name: str, eval_cases: list[dict], top_k: int = 10) -> tuple[dict, pd.DataFrame]:
    queries = [case["query"] for case in eval_cases]
    if not queries:
        queries = [QUERY]

    result_rows = []
    latency_values = []
    for query in queries:
        search_engine = SearchEngine(model_name)
        results, latency_ms = search_engine.search(query, top_k=top_k)
        latency_values.append(latency_ms)
        
        
        for _, row in results.iterrows():
            
            result_rows.append({
                "model": model_name,
                "query": query,
                "rank": int(row["rank"]),
                "tmdb_id": row["tmdb_id"],
                "movie_id": row["movie_id"],
                "title": row["title"],
                "score": float(row["score"]),
            })

    details = pd.DataFrame(result_rows)

    eval = EvaluationMetrics()
    
    metrics = eval.evaluate_results(
        details,
        eval_cases,
        ConfigModel.EVAL_TOP_K,
    ) if eval_cases else {}
    
    vector_store = VectorDbStorage(model_name)
    vector_store.load()
    
    metrics.update({
        "model": model_name,
        "catalog_count": len(catalog),
        "embedding_dimension": vector_store.embeddings.shape[1],
        "average_query_latency_ms": float(np.mean(latency_values)),
        "search_coverage": float(details["tmdb_id"].nunique() / max(len(catalog), 1)),
        "average_unique_results": float(details.groupby("query")["tmdb_id"].nunique().mean()),
    })
    return metrics, details


benchmark_metrics = []
benchmark_details = []
for model_name in ConfigModel.MODEL_NAMES:
    metrics, details = benchmark_model(model_name, EVAL_CASES, ConfigModel.TOP_K)
    benchmark_metrics.append(metrics)
    benchmark_details.append(details)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1993.93it/s]


[INFO] Tìm kiếm hoàn tất trong 0.04 giây


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1062.86it/s]


[INFO] Tìm kiếm hoàn tất trong 0.02 giây


In [125]:
benchmark_metrics_df = pd.DataFrame(benchmark_metrics)
benchmark_results_df = pd.concat(benchmark_details, ignore_index=True)
display(benchmark_metrics_df.head(10))
display(benchmark_results_df.head(10))

,query_count,precision_at_k,recall_at_k,mrr,ndcg_at_k,model,catalog_count,embedding_dimension,average_query_latency_ms,search_coverage,average_unique_results
0,1,0.0,0.0,0.0,0.0,sentence-transformers/paraphrase-multilingual-...,19706,384,0.035164,0.000507,10.0
1,1,0.0,0.0,0.0,0.0,sentence-transformers/distiluse-base-multiling...,19706,512,0.018157,0.000507,10.0


,model,query,rank,tmdb_id,movie_id,title,score
0,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,1,2133.0,3755.0,Cơn Bão Kinh Hoàng,0.590673
1,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,2,2787.0,3300.0,Riddick: Hành Tinh Tăm Tối,0.589344
2,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,3,7450.0,3745.0,Titan A.E.,0.582167
3,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,4,955.0,3623.0,Nhiệm Vụ: Bất Khả Thi 2,0.573591
4,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,5,79782.0,895.0,Wenecja,0.558353
5,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,6,1907.0,3285.0,The Beach,0.527930
6,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,7,47724.0,3580.0,Up at the Villa,0.523810
7,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,8,9679.0,3717.0,Biến Mất Trong 60 Giây,0.509021
8,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,9,4247.0,3785.0,Liên Khúc Phim Kinh Dị,0.505652
9,sentence-transformers/paraphrase-multilingual-...,phim phiêu lưu vũ trụ,10,13539.0,3453.0,Here on Earth,0.504524


## Save experiment artifacts

Embeddings được lưu riêng theo model để có thể nạp lại mà không encode lại catalog. Metadata và config giúp truy vết model, số chiều vector, text input và thời gian encode.

In [126]:
run_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
benchmark_metrics_path = ConfigModel.ARTIFACT_DIR / "embedding_benchmark.csv"
benchmark_results_path = ConfigModel.ARTIFACT_DIR / f"search_results_{run_id}.csv"
run_config_path = ConfigModel.ARTIFACT_DIR / f"run_config_{run_id}.json"

benchmark_metrics_df.to_csv(benchmark_metrics_path, index=False, encoding="utf-8-sig")
benchmark_results_df.to_csv(benchmark_results_path, index=False, encoding="utf-8-sig")
run_config_path.write_text(
    json.dumps(
        {
            "run_id": run_id,
            "catalog_path": str(ConfigModel.CATALOG_PATH),
            "catalog_count": len(catalog),
            "model_names": ConfigModel.MODEL_NAMES,
            "top_k": ConfigModel.TOP_K,
            "eval_top_k": ConfigModel.EVAL_TOP_K,
            "evaluation_cases": EVAL_CASES,
            "search_id_column": SEARCH_ID_COLUMN,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Metrics: {benchmark_metrics_path}")
print(f"Search results: {benchmark_results_path}")
print(f"Run config: {run_config_path}")

Metrics: ..\data\artifacts\embeddings\embedding_benchmark.csv
Search results: ..\data\artifacts\embeddings\search_results_20260915T165745Z.csv
Run config: ..\data\artifacts\embeddings\run_config_20260915T165745Z.json
